In [1]:
import os
import json
from tkinter.font import names
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

In [2]:

project_root = os.path.dirname(os.getcwd())

In [3]:
processed_resume = os.path.join(project_root, r"src\ph_resume_ext\resume\processed")
golden_resume = os.path.join(project_root, r"src\ph_resume_ext\resume\Golden_Records")


In [4]:
def evaluate_field(pred, truth):
        return int(pred.strip().lower() == truth.strip().lower())


In [5]:
def evaluate_skills(pred, truth):
        pred_set = set([s.lower() for s in pred])
        truth_set = set([s.lower() for s in truth])
        tp = len(pred_set & truth_set)
        fp = len(pred_set - truth_set)
        fn = len(truth_set - pred_set)

        precision = tp / (tp+fp) if tp+fp else 0
        recall = tp / (tp+fn) if tp+fn else 0
        f1 = 2*precision*recall/(precision+recall) if precision+recall else 0
        return precision, recall, f1

In [6]:
def load_processed():
    """Load processed resumes into state"""
    processed_resume_paths = [
        os.path.join(processed_resume, f)
        for f in os.listdir(processed_resume)
        if f.endswith(".json")
    ]
    return processed_resume_paths

In [7]:
load_processed()

['c:\\resume_extration_crewai_flow\\ph_resume_ext\\src\\ph_resume_ext\\resume\\processed\\elegant-ms-word-resume-template.json',
 'c:\\resume_extration_crewai_flow\\ph_resume_ext\\src\\ph_resume_ext\\resume\\processed\\entry-level-data-scientist-resume-example.json',
 'c:\\resume_extration_crewai_flow\\ph_resume_ext\\src\\ph_resume_ext\\resume\\processed\\official-ms-word-resume-template.json',
 'c:\\resume_extration_crewai_flow\\ph_resume_ext\\src\\ph_resume_ext\\resume\\processed\\senior-data-scientist-resume-example.json',
 'c:\\resume_extration_crewai_flow\\ph_resume_ext\\src\\ph_resume_ext\\resume\\processed\\standout-ms-word-resume-template.json']

In [8]:
def load_golden():
    """Load golden resumes into state"""
    golden_resume_paths = [
        os.path.join(golden_resume, f)
        for f in os.listdir(golden_resume)
        if f.endswith(".json")
    ]
    return golden_resume_paths

In [9]:
load_golden()

['c:\\resume_extration_crewai_flow\\ph_resume_ext\\src\\ph_resume_ext\\resume\\Golden_Records\\elegant-ms-word-resume-template.json',
 'c:\\resume_extration_crewai_flow\\ph_resume_ext\\src\\ph_resume_ext\\resume\\Golden_Records\\entry-level-data-scientist-resume-example.json',
 'c:\\resume_extration_crewai_flow\\ph_resume_ext\\src\\ph_resume_ext\\resume\\Golden_Records\\official-ms-word-resume-template.json',
 'c:\\resume_extration_crewai_flow\\ph_resume_ext\\src\\ph_resume_ext\\resume\\Golden_Records\\senior-data-scientist-resume-example.json',
 'c:\\resume_extration_crewai_flow\\ph_resume_ext\\src\\ph_resume_ext\\resume\\Golden_Records\\standout-ms-word-resume-template.json']

In [10]:
def evaluate():
        """Evaluate processed vs golden records"""
        # build golden dict
        golden_data = {}
        for gfile in load_golden():
            name = os.path.splitext(os.path.basename(gfile))[0]
            with open(gfile, "r", encoding="utf-8") as f:
                golden_data[name] = json.load(f)

        summary = []

        for pfile in load_processed():
            name = os.path.splitext(os.path.basename(pfile))[0]
            with open(pfile, "r", encoding="utf-8") as f:
                pred = json.load(f)

            truth = golden_data.get(name)
            if not truth:
                continue
            
            def get_field(d, *names):
                for name in names:
                    for k in d.keys():
                        if k.lower() == name.lower():
                            return d[k]
                return ""

            name_acc = evaluate_field(get_field(pred, "First_Name"), get_field(truth, "First_Name"))
            lname_acc = evaluate_field(get_field(pred, "Last_Name"), get_field(truth, "Last_Name"))
            email_acc = evaluate_field(get_field(pred, "Email_Address", "email_address"), get_field(truth, "Email_Address", "email_address"))
            p, r, f1 =evaluate_skills(get_field(pred, "Skills", "skills", []), get_field(truth, "Skills", "skills", []))

            summary.append({
                "resume": name,
                "first_name_acc": name_acc,
                "last_name_acc": lname_acc,
                "email_acc": email_acc,
                "skills_precision": p,
                "skills_recall": r,
                "skills_f1": f1
            })

        df = pd.DataFrame(summary)
        print("\nOverall F1 (skills):", df["skills_f1"].mean())
        output_csv = os.path.join(processed_resume, "evaluation_results.csv")
        df.to_csv(output_csv, index=False, encoding="utf-8")
        return df

In [11]:
evaluate()


Overall F1 (skills): 0.9178571428571429


,resume,first_name_acc,last_name_acc,email_acc,skills_precision,skills_recall,skills_f1
0,elegant-ms-word-resume-template,1,1,1,1.000000,1.000000,1.000000
1,entry-level-data-scientist-resume-example,1,1,1,1.000000,1.000000,1.000000
2,official-ms-word-resume-template,1,1,1,1.000000,1.000000,1.000000
3,senior-data-scientist-resume-example,1,1,1,0.875000,0.875000,0.875000
4,standout-ms-word-resume-template,1,1,1,0.714286,0.714286,0.714286
